# Phase 9 & 10: Model Training (Baseline & CNN)

In this notebook, we pull our subset from memory, normalize it, split it into Training and Validation sets, and train two models:
1. A Baseline (Logistic Regression) to establish a minimum performance bar.
2. A Tiny CNN (Convolutional Neural Network) built in PyTorch.

In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import medmnist
from medmnist import INFO

SEED = 42
SAMPLES_PER_CLASS = 50

### 1. Load Data, Create Subset, & Normalize
We repeat the fast in-memory load to ensure this notebook can be run independently.

In [2]:
info = INFO['pneumoniamnist']
DataClass = getattr(medmnist, info['python_class'])
dataset = DataClass(split='train', download=True, root='../data/')

all_images = dataset.imgs
all_labels = dataset.labels.squeeze()

np.random.seed(SEED)
subset_indices = []
for class_id in [0, 1]:
    class_idx = np.where(all_labels == class_id)[0]
    subset_indices.extend(np.random.choice(class_idx, SAMPLES_PER_CLASS, replace=False))

# Convert to float32 and normalize pixel values to [0, 1] for ML models
X = all_images[subset_indices].astype(np.float32) / 255.0
y = all_labels[subset_indices]

# Split Data (80% Train, 20% Val)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

Training set: 80 samples
Validation set: 20 samples


### 2. Train Baseline (Logistic Regression)

In [3]:
# Flatten images for Logistic Regression (from 28x28 to 784 1D array)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)

baseline_model = LogisticRegression(random_state=SEED, max_iter=1000)
baseline_model.fit(X_train_flat, y_train)

val_preds = baseline_model.predict(X_val_flat)
baseline_acc = accuracy_score(y_val, val_preds)
print(f"Baseline Validation Accuracy: {baseline_acc * 100:.2f}%")

Baseline Validation Accuracy: 85.00%


### 3. Train Tiny CNN (PyTorch)
Now we build a small CNN to see if spatial feature extraction outperforms the baseline.

In [4]:
# Convert to PyTorch Tensors (Add Channel dimension for CNN: N x C x H x W)
X_train_t = torch.tensor(X_train).unsqueeze(1)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_t = torch.tensor(X_val).unsqueeze(1)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=True)

class TinyCNN(nn.Module):
    def __init__(self):
        super(TinyCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(16 * 7 * 7, 1) # 28 -> 14 -> 7
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.sigmoid(self.fc(x))
        return x

torch.manual_seed(SEED)
cnn_model = TinyCNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

epochs = 20
for epoch in range(epochs):
    cnn_model.train()
    epoch_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = cnn_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss/len(train_loader):.4f}")

Epoch 5/20, Loss: 0.6680
Epoch 10/20, Loss: 0.5980
Epoch 15/20, Loss: 0.4900


Epoch 20/20, Loss: 0.3988


### 4. Evaluate CNN & Save Model
Let's see how the CNN performs on the validation set, and export it for the FastAPI backend!

In [5]:
cnn_model.eval()
with torch.no_grad():
    val_outputs = cnn_model(X_val_t)
    val_preds = (val_outputs >= 0.5).float()
    cnn_acc = (val_preds == y_val_t).float().mean().item()

print(f"CNN Validation Accuracy: {cnn_acc * 100:.2f}%")

# Save the model weights to our models directory
os.makedirs('../models', exist_ok=True)
torch.save(cnn_model.state_dict(), '../models/pneumonia_classifier_v1.pt')
print("✅ Model saved to 'ml/models/pneumonia_classifier_v1.pt'")

CNN Validation Accuracy: 65.00%
✅ Model saved to 'ml/models/pneumonia_classifier_v1.pt'
